# Guardrails V2 + V3 walkthrough (D1 + D2 + D2b + D2c)

This notebook walks through the deterministic and LLM-audited guardrail layers added in **D1 (Guardrails V2)**, **D2 (Guardrails V3)**, **D2b (cross-intent support-status fix)**, and **D2c (evidence-packet truncation fix)**, cell by cell, using the same backend services that power the app. It picks up where [03_backend_query_answer_walkthrough.ipynb](03_backend_query_answer_walkthrough.ipynb) leaves off: that notebook covers extraction, RxNorm resolution, and retrieval; this one covers what happens to the *generated answer* after that — the coverage-driven contract, post-generation validation, per-claim support status, and the optional LLM critic with bounded regeneration.

**D1 — Guardrails V2** (`src/query_answer/contract.py`, `src/query_answer/validation.py`):
- `build_answer_contract(understanding, coverage)` turns the deterministic coverage report into a list of `must_mention` / `must_caveat` `AnswerContractItem`s, plus an answer-level `coverage_level` (`direct`/`partial`/`limited`/`none`). Each intent item's `required_sections` is now the section(s) *actually matched* for that intent (`EvidenceCoverageItem.matched_sections`), not a static allow-list (D2b).
- The contract is fed into the synthesis prompt as `answer_contract` (see `EvidenceAnswerSynthesizer.build_evidence_packet`).
- `validate_and_enforce(answer, contract)` runs after synthesis: deterministically appends any `must_caveat` the model dropped, flags yes/no medical-advice framing, and records unaddressed `must_mention` topics as `ValidationFinding`s.

**D2 — Guardrails V3** (`src/query_answer/critic.py`):
- `deterministic_support_status(...)` / `apply_deterministic_statuses(...)` classify every bullet's `support_status` (`strong`/`partial`/`limited`/`none`) from its citations against the contract. This is the always-on floor — it runs even when no LLM critic is configured (e.g. demo mode).
- `critique_answer(...)` is the optional LLM critic: it reads each numbered claim against the same evidence packet used for synthesis and may override the deterministic status, flag issues, and request regeneration.
- `finalize_answer_critique(...)` is the orchestrator wired into `QueryAnswerService.answer()`: applies the deterministic floor, runs the critic if `enable_answer_critic` is on, and performs **at most one** feedback-driven regeneration.

**D2b — fix cross-intent support-status leakage** (`src/query_answer/critic.py`, `src/query_answer/synthesizer.py`): found by running this exact notebook against *"Can I take ibuprofen for my migraine if I'm allergic to aspirin?"* — a bullet stating no `drug_interactions` text was retrieved was inconsistently scored `strong` vs `limited` across runs, purely depending on which section the LLM happened to cite. Root cause: the floor pooled `required_sections` across *every* addressed intent into one flat set, so a citation backing `allergy_context_check` (e.g. `warnings`) could spuriously "support" an unrelated `interaction_check` bullet. Fix: each bullet is now tagged with the contract `topic` it addresses (whitelisted against the contract, same pattern as citation whitelisting — see `EvidenceBullet.topic`), and `deterministic_support_status` scopes its section check to that topic's own `required_sections` when a match exists, falling back to the legacy pooled check (`critic._addressed_intent_sections`) for untagged bullets. See section 10 below and the "Update" note at the end of section 14.

**D2c — fix evidence-packet truncation starving later label sections** (`src/query_answer/synthesizer.py`): found while investigating D2b — even after that fix, the *same* query kept reporting "the packet does not include any drug_interactions text," and digging into section 7's evidence packet below showed why: `label_section_payloads()` truncated per drug to a flat 16-entry cap walked in a fixed section priority order, with no per-section sub-limit and no deduplication. Ibuprofen alone has 17+ non-empty entries across `boxed_warning`/`contraindications`/`warnings` (many near-identical "allergy alert" boilerplate from different manufacturer label records) — enough to exhaust the cap before `drug_interactions` was ever reached, even though the contract had already marked `interaction_check` "addressed." Fix: deduplicate near-identical boilerplate per section, cap each section to `MAX_SECTION_ENTRIES`, and guarantee any section the contract relies on (`required_label_sections`, shared between `contract.py` and `critic.py`) survives the overall cap. See section 7 below and the "Update" note at the end of section 14.

This notebook intentionally calls a couple of private helpers (`critic._addressed_intent_sections`, `critic._has_unresolved_gap`, `critic._topic_required_sections`) to make the support-status classification legible. That's for inspection only — production code should keep using the public functions. Section 14 collects gaps and edge cases worth a second look.

## 0. Setup

Run this notebook from the project root or directly from `notebooks/`. All live-LLM toggles below default to `False`, so running the whole notebook top to bottom makes no paid API calls — every guardrail step is demonstrated with hand-built mock answers and mock critic responses first, with an optional live cell after each one.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repository root.")


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

print(f"Repository root: {ROOT}")

Repository root: /Users/Marie_Cordes/code/mariecordes/rx-ray


In [2]:
from src.dossier.builder import DossierBuilder
from src.dossier.openfda_store import OpenFDALabelStore
from src.dossier.rxnorm_store import RxNormParquetStore
from src.query_answer import critic
from src.query_answer.config import QueryAnswerParameters, load_query_answer_parameters
from src.query_answer.context import (
    build_context_targeted_evidence,
    merge_context_evidence_into_secondary,
    merge_context_evidence_into_understanding,
)
from src.query_answer.contract import build_answer_contract
from src.query_answer.coverage import build_evidence_coverage
from src.query_answer.critic import (
    apply_critique_statuses,
    apply_deterministic_statuses,
    critique_answer,
    finalize_answer_critique,
)
from src.query_answer.secondary import build_secondary_evidence
from src.query_answer.service import QueryAnswerService
from src.query_answer.synthesizer import EvidenceAnswerSynthesizer
from src.query_answer.validation import validate_and_enforce
from src.query_understanding.extractor import HybridQueryExtractor
from src.query_understanding.service import QueryUnderstandingService


def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    return value


def show_json(value):
    display(JSON(to_jsonable(value)))


def as_frame(rows):
    return pd.DataFrame([to_jsonable(row) for row in rows])

## 1. Choose a query and runtime toggles

This is the same multi-drug, multi-intent query used in the demo fixture and in [04_interaction_evidence_map_debug.ipynb](04_interaction_evidence_map_debug.ipynb): a current medication (cetirizine), two mentioned drugs (ibuprofen, aspirin) that trigger `interaction_check`, plus an allergy and a context phrase that trigger context-targeted lookups. It's a good stress test for the contract because it should produce both `must_mention` and `must_caveat` items across more than one intent.

Flip `RUN_LIVE_ANSWER_SYNTHESIS` / `RUN_LIVE_CRITIC` to `True` only when you want to spend real API calls against the configured models. Leave `ENABLE_ANSWER_CRITIC_OVERRIDE = None` to use whatever `conf/base/parameters.yml` says (`enable_answer_critic`, off by default); set it to `True`/`False` to force one path for this notebook run only.

In [3]:
QUERY = (
    # "I currently take cetirizine for my pollen allergy. can i take both "
    # "ibuprofen and aspirin against swollen eyes?"
    "Can I take ibuprofen for my migraine if I'm allergic to aspirin?"
)

RUN_LIVE_ANSWER_SYNTHESIS = True
RUN_LIVE_CRITIC = True
ENABLE_ANSWER_CRITIC_OVERRIDE: bool | None = None

base_parameters = load_query_answer_parameters()
if ENABLE_ANSWER_CRITIC_OVERRIDE is None:
    parameters = base_parameters
else:
    parameters = base_parameters.__class__(
        **{
            **base_parameters.__dict__,
            "enable_answer_critic": ENABLE_ANSWER_CRITIC_OVERRIDE,
        }
    )

print(f"Query: {QUERY}")
print("enable_answer_critic:", parameters.enable_answer_critic)
print("critic_max_regenerations:", parameters.critic_max_regenerations)
print(
    "Answer-synthesis LLM configured:",
    bool(os.getenv("ANSWER_SYNTHESIS_OPENAI_API_KEY") and os.getenv("ANSWER_SYNTHESIS_OPENAI_MODEL")),
)
print("Critic LLM falls back to the same env vars unless ANSWER_CRITIC_OPENAI_* is set.")

Query: Can I take ibuprofen for my migraine if I'm allergic to aspirin?
enable_answer_critic: False
critic_max_regenerations: 1
Answer-synthesis LLM configured: True
Critic LLM falls back to the same env vars unless ANSWER_CRITIC_OPENAI_* is set.


## 2. Instantiate the backend services

Same request-time stack as the FastAPI app: RxNorm parquet files plus OpenFDA live lookup with a local cache.

In [4]:
rxnorm_store = RxNormParquetStore()
openfda_store = OpenFDALabelStore(use_cache=True, allow_live=True)
builder = DossierBuilder(rxnorm_store=rxnorm_store, openfda_store=openfda_store)

extractor = HybridQueryExtractor()
understanding_service = QueryUnderstandingService(builder=builder, extractor=extractor)
synthesizer = EvidenceAnswerSynthesizer()
query_answer_service = QueryAnswerService(
    builder=builder,
    understanding_service=understanding_service,
    synthesizer=synthesizer,
)

print("Services ready.")

Services ready.


## 3. Run query understanding

Same object `POST /query-understanding` returns: extraction, RxNorm resolution, and the primary dossier. See notebook 03 for a deeper look at each step inside this call.

In [5]:
understanding = understanding_service.understand(
    QUERY,
    openfda_limit=parameters.default_openfda_limit,
)

print("Extraction mode:", understanding.extraction_mode)
print("Warnings:", understanding.warnings)
print("Errors:", understanding.errors)
show_json(understanding.state)

Extraction mode: hybrid
Warnings: []
Errors: []


<IPython.core.display.JSON object>

## 4. Build secondary and context-targeted evidence

This mirrors `QueryAnswerService.answer()`: secondary evidence for the other mentioned/current drugs, then context-targeted lookups for the allergy/condition/patient-context state, merged back into `understanding` and `secondary_evidence`. The deterministic coverage report and the contract both depend on having this evidence in hand first.

In [6]:
secondary_evidence = build_secondary_evidence(understanding, builder, parameters)
context_evidence = build_context_targeted_evidence(
    understanding, secondary_evidence, builder, parameters
)
understanding = merge_context_evidence_into_understanding(understanding, context_evidence)
secondary_evidence = merge_context_evidence_into_secondary(secondary_evidence, context_evidence)

display(
    as_frame(
        [
            {
                "mention_text": item.mention_text,
                "role": item.role,
                "resolved_name": item.resolved_concept.name,
                "labels_found": item.label_evidence.labels_found if item.label_evidence else 0,
                "interaction_labels_found": (
                    item.interaction_label_evidence.labels_found
                    if item.interaction_label_evidence
                    else 0
                ),
                "retrieval_modes": item.retrieval_modes,
            }
            for item in secondary_evidence
        ]
    )
)
display(
    as_frame(
        [
            {
                "target_label": item.target_label,
                "target_category": item.target_category,
                "resolved_name": item.resolved_concept.name,
                "searched_fields": item.searched_fields,
                "labels_found": item.label_evidence.labels_found if item.label_evidence else 0,
            }
            for item in context_evidence
        ]
    )
)

,mention_text,role,resolved_name,labels_found,interaction_labels_found,retrieval_modes
0,aspirin,allergy,aspirin,12,6,"[standard_secondary_label_lookup, cache, inter..."


,target_label,target_category,resolved_name,searched_fields,labels_found
0,migraine,condition,ibuprofen,"[indications_and_usage, warnings, contraindica...",3
1,migraine,condition,aspirin,"[indications_and_usage, warnings, contraindica...",3


## 5. Deterministic coverage report — `build_evidence_coverage`

This is the input to D1's contract builder: did retrieval actually find evidence for each extracted state item and intent? Each row's `status` is one of `addressed` / `not_found_in_evidence` / `not_retrieved` / `out_of_scope`.

In [7]:
coverage = build_evidence_coverage(
    understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
)

print("Summary counts:", coverage.summary_counts)
as_frame(coverage.items)

Summary counts: {'addressed': 6, 'not_found_in_evidence': 0, 'not_retrieved': 0, 'out_of_scope': 0}


,category,label,status,reason,matched_evidence,source_id,section,matched_sections,target_rxcui
0,primary_drug,ibuprofen,addressed,Drug label evidence was retrieved for ibuprofen.,NaN,NaN,NaN,[],5640
1,mentioned_drug,aspirin,addressed,Drug label evidence was retrieved for aspirin.,NaN,NaN,NaN,[],1191
2,allergy,aspirin,addressed,The retrieved label text explicitly mentions t...,"...patients who have experienced asthma, urtic...",5e595111-8477-4ec7-8d32-a0aeabc04e1a,contraindications,[],NaN
3,condition,migraine,addressed,The retrieved label text explicitly mentions t...,...right before or after heart surgery Ask a d...,460cceff-0b2e-cbfe-e063-6394a90aa46f,warnings,[],5640
4,intent,allergy_context_check,addressed,"Retrieved label sections (contraindications, w...",...CONTRAINDICATIONS Ibuprofen tablets are con...,5e595111-8477-4ec7-8d32-a0aeabc04e1a,contraindications,"[contraindications, warnings]",5640
5,intent,interaction_check,addressed,Drug-interactions label text was retrieved for...,...Preexisting asthma Patients with asthma may...,5e595111-8477-4ec7-8d32-a0aeabc04e1a,drug_interactions,[drug_interactions],5640


## 6. Build the answer contract — `build_answer_contract` (D1, new in this work)

Turns the coverage report into a checklist synthesis must satisfy: `must_mention` items name a topic with evidence available; `must_caveat` items are statements that must appear in `limitations`, verbatim or rephrased. Watch for the always-present `interaction_terminology` caveat whenever `interaction_check` is one of the extracted intents — it's emitted regardless of how good the interaction evidence is, which matters for how D2 classifies support status later (see section 10).

In [8]:
contract = build_answer_contract(understanding, coverage)

print("Coverage level:", contract.coverage_level)
as_frame(contract.items)

Coverage level: direct


,kind,topic,intent,statement,evidence_available,required_sections,coverage_category,coverage_label,target_rxcui
0,must_mention,allergy_context_check,allergy_context_check,Address what the retrieved allergy label secti...,True,"[contraindications, warnings]",intent,allergy_context_check,None
1,must_mention,interaction_check,interaction_check,Address what the retrieved drug interaction la...,True,[drug_interactions],intent,interaction_check,None
2,must_caveat,interaction_terminology,interaction_check,RxNorm terminology overlap is not evidence of ...,True,[],intent,interaction_check,None


## 7. Confirm the contract reaches the synthesis evidence packet

`EvidenceAnswerSynthesizer.build_evidence_packet` embeds the contract under `answer_contract`, alongside the non-citable `label_product_context` block (product/formulation info that must never satisfy a citation requirement). `label_sections` itself is bounded by `label_section_payloads()` (D2c): it deduplicates near-identical boilerplate repeated across a drug's many label records, caps each section to `MAX_SECTION_ENTRIES`, and guarantees any section the contract already relies on (`required_label_sections(contract)`) survives the overall `MAX_LABEL_SECTIONS` cap. The per-section breakdown below should always include `drug_interactions` for this query — before D2c it didn't, because ibuprofen alone has 17+ non-empty entries in earlier-priority sections.

In [9]:
evidence_packet = synthesizer.build_evidence_packet(
    understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
    contract=contract,
)

print("Top-level keys:", list(evidence_packet.keys()))
print("label_sections:", len(evidence_packet.get("label_sections", [])))
print("label_product_context:", len(evidence_packet.get("label_product_context", [])))

section_counts = (
    pd.Series(
        [
            (entry["evidence_scope"], entry["drug_name"], entry["section"])
            for entry in evidence_packet.get("label_sections", [])
        ]
    )
    .value_counts()
    .rename("entries")
)
display(section_counts)
print(
    "drug_interactions present in label_sections?",
    any(
        entry["section"] == "drug_interactions"
        for entry in evidence_packet.get("label_sections", [])
    ),
)
show_json(evidence_packet["answer_contract"])

Top-level keys: ['query', 'state', 'resolved_primary_drug', 'label_evidence_scope', 'ingredient_fallback', 'rxnorm_relationship_summary', 'secondary_drug_evidence', 'context_targeted_evidence', 'label_sources', 'label_sections', 'label_product_context', 'answer_contract', 'retrieval_notes']
label_sections: 32
label_product_context: 12


(primary, None, warnings)                  3
(primary, None, pregnancy)                 3
(primary, None, indications_and_usage)     3
(secondary, aspirin, boxed_warning)        3
(secondary, aspirin, contraindications)    3
(secondary, aspirin, warnings)             3
(secondary, aspirin, drug_interactions)    3
(secondary, aspirin, pregnancy)            3
(primary, None, drug_interactions)         2
(primary, None, lactation)                 2
(primary, None, boxed_warning)             1
(primary, None, contraindications)         1
(primary, None, adverse_reactions)         1
(secondary, aspirin, lactation)            1
Name: entries, dtype: int64

drug_interactions present in label_sections? True


<IPython.core.display.JSON object>

## 8. Draft an answer to audit

To keep the rest of this notebook free of paid API calls by default, this cell builds a deliberately imperfect mock answer: it cites one real, allowed section, but skips at least one `must_caveat` from the contract so the next section has something real to enforce. Flip `RUN_LIVE_ANSWER_SYNTHESIS` in section 1 to use the real synthesizer call instead.

In [10]:
from src.query_answer.models import EvidenceAnswer, EvidenceBullet, EvidenceCitation

allowed_citations = synthesizer.allowed_citations(evidence_packet)
first_citation = next(iter(sorted(allowed_citations)), None)

mock_answer = EvidenceAnswer(
    response="The retrieved labels mention an interaction-style caution for these medications.",
    evidence_summary="Demo answer built by hand to exercise validate_and_enforce.",
    summary="Demo answer.",
    bullets=[
        EvidenceBullet(
            text="A retrieved label section mentions the other medication.",
            citations=(
                [EvidenceCitation(source_id=first_citation[0], section=first_citation[1])]
                if first_citation
                else []
            ),
        ),
    ],
    limitations=[],
    safety_note="This is an educational summary of retrieved public evidence, not medical advice.",
)

if RUN_LIVE_ANSWER_SYNTHESIS:
    synthesis_result = synthesizer.synthesize(
        QUERY,
        understanding,
        secondary_evidence=secondary_evidence,
        context_evidence=context_evidence,
        contract=contract,
    )
    print("Warnings:", synthesis_result.warnings)
    print("Errors:", synthesis_result.errors)
    draft_answer = synthesis_result.answer or mock_answer
else:
    print("Skipped live answer synthesis; using the hand-built mock answer.")
    draft_answer = mock_answer

show_json(draft_answer)

Warnings: []
Errors: []


<IPython.core.display.JSON object>

## 9. Validate and enforce the contract — `validate_and_enforce` (D1)

Runs after synthesis: appends any `must_caveat` missing from `limitations`, flags yes/no medical-advice framing, and records unaddressed `must_mention` topics as findings. Compare `limitations` before and after — anything new was added deterministically, not by the model.

In [11]:
validated_answer, validation = validate_and_enforce(draft_answer, contract)

print("Limitations before:", draft_answer.limitations)
print("Limitations after: ", validated_answer.limitations)
show_json(validation)

Limitations before: ['The retrieved labels do not provide enough information to determine your specific risk from the phrase “allergic to aspirin” alone; the label language focuses on prior asthma/urticaria/allergic-type reactions after aspirin or other NSAIDs.', 'RxNorm terminology overlap is not evidence of a clinical interaction.']
Limitations after:  ['The retrieved labels do not provide enough information to determine your specific risk from the phrase “allergic to aspirin” alone; the label language focuses on prior asthma/urticaria/allergic-type reactions after aspirin or other NSAIDs.', 'RxNorm terminology overlap is not evidence of a clinical interaction.']


<IPython.core.display.JSON object>

## 10. Deterministic support-status floor — `apply_deterministic_statuses` (D2, topic-scoped since D2b)

This always runs, with or without the LLM critic. For each bullet: `none` if uncited. Otherwise, the cited sections are checked against an "addressed sections" set — scoped to the *bullet's own topic* when `bullet.topic` resolves to a section-bearing intent item on the contract (D2b), or the legacy pooled set across every addressed intent when it doesn't. `limited` if the cited sections don't overlap that set, `strong` if they do and there's no unresolved contract gap, `partial` if they do but a gap remains (the always-present `interaction_terminology` caveat is excluded from that gap check — see `critic.STRUCTURAL_CAVEAT_TOPICS`). The next cell exposes the private helpers that drive this classification for the first bullet, for inspection.

In [12]:
floored_answer = apply_deterministic_statuses(validated_answer, contract)

display(
    as_frame(
        [
            {
                "bullet_index": index,
                "text": bullet.text,
                "topic": bullet.topic,
                "cited_sections": [c.section for c in bullet.citations],
                "support_status": bullet.support_status,
            }
            for index, bullet in enumerate(floored_answer.bullets)
        ]
    )
)

,bullet_index,text,topic,cited_sections,support_status
0,0,Ibuprofen labels include an allergy alert that...,allergy_context_check,[warnings],strong
1,1,The contraindications section states ibuprofen...,allergy_context_check,[contraindications],strong
2,2,The retrieved ibuprofen label text includes a ...,interaction_check,[drug_interactions],strong


In [13]:
if floored_answer.bullets:
    sample = floored_answer.bullets[0]
    topic_sections = critic._topic_required_sections(sample.topic, contract)
    print("Sample bullet:", sample.text)
    print("Cited sections:", {c.section for c in sample.citations})
    print("Bullet topic:", sample.topic)
    if topic_sections is not None:
        print("Topic-scoped required sections (D2b):", topic_sections)
    else:
        print(
            "No topic match on the contract -- falling back to the legacy "
            "pooled set:",
            critic._addressed_intent_sections(contract),
        )
    print("Contract still has an unresolved (non-structural) gap:", critic._has_unresolved_gap(contract))
    print("=> support_status:", sample.support_status)
else:
    print("No bullets to inspect.")

Sample bullet: Ibuprofen labels include an allergy alert that it may cause a severe allergic reaction, especially in people allergic to aspirin, with symptoms such as hives, facial swelling, asthma/wheezing, shock, rash, and blisters; if an allergic reaction occurs, stop use and seek medical help.
Cited sections: {'warnings'}
Bullet topic: allergy_context_check
Topic-scoped required sections (D2b): {'contraindications', 'warnings'}
Contract still has an unresolved (non-structural) gap: False
=> support_status: strong


## 11. Optional LLM critic — `critique_answer` (D2, new)

The critic reads each numbered claim against the same evidence packet and may override the deterministic status, flag issues (`unsupported`/`overconfident`/`missing_caveat`/`yes_no_framing`/`invented_citation`), and request regeneration via `needs_regeneration`. The mock requester below stands in for an LLM response so this runs with no API calls; flip `RUN_LIVE_CRITIC` in section 1 to call the real model instead.

In [14]:
def mock_critic_requester(*, messages, prompt_config):
    """Stands in for the critic LLM call (tests use the same pattern)."""
    return {
        "claims": [
            {
                "index": 0,
                "support_status": "partial",
                "rationale": "Cited, but the interaction caveat is always present for this intent.",
                "issues": [],
            }
        ],
        "global_findings": [],
        "needs_regeneration": False,
    }


if RUN_LIVE_CRITIC and critic._critic_llm_configured():
    critic_requester = None  # None => critique_answer makes a real API call
else:
    if RUN_LIVE_CRITIC:
        print("RUN_LIVE_CRITIC is True but no critic/synthesis LLM is configured; using the mock instead.")
    critic_requester = mock_critic_requester

outcome = critique_answer(
    query=QUERY,
    answer=floored_answer,
    evidence_packet=evidence_packet,
    json_requester=critic_requester,
)

print("needs_regeneration:", outcome.needs_regeneration)
show_json(outcome.critique)

critiqued_answer = apply_critique_statuses(floored_answer, outcome.critique)
[b.support_status for b in critiqued_answer.bullets]

needs_regeneration: True


<IPython.core.display.JSON object>

['strong', 'strong', 'strong']

## 12. Full orchestration — `finalize_answer_critique` (D2 entrypoint)

This is the function `QueryAnswerService.answer()` calls. Two runs below show both branches without spending real tokens:

**(a) critic disabled** — deterministic floor only, `critique.enabled` is `False`.

**(b) critic enabled, forced regeneration** — a mock critic that says `needs_regeneration=True` on its first call and `False` on the recheck after regeneration, paired with a mock synthesis requester for the regeneration call. This is the same pattern `tests/test_critic.py::test_finalize_answer_critique_regenerates_exactly_once` uses to prove regeneration happens **exactly once**, never in a loop.

In [15]:
deterministic_only_parameters = parameters.__class__(
    **{**parameters.__dict__, "enable_answer_critic": False}
)

answer_a, critique_a, validation_a = finalize_answer_critique(
    query=QUERY,
    understanding=understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
    answer=draft_answer,
    contract=contract,
    validation=validation,
    synthesizer=synthesizer,
    parameters=deterministic_only_parameters,
)

print("(a) critic.enabled:", critique_a.enabled, "| source:", critique_a.source)
[b.support_status for b in answer_a.bullets]

(a) critic.enabled: False | source: deterministic


['strong', 'strong', 'strong']

In [16]:
regeneration_critic_calls = {"count": 0}


def forced_regeneration_critic(*, messages, prompt_config):
    regeneration_critic_calls["count"] += 1
    first_call = regeneration_critic_calls["count"] == 1
    return {
        "claims": [
            {
                "index": 0,
                "support_status": "limited" if first_call else "strong",
                "rationale": "Forced for this walkthrough.",
                "issues": ["unsupported"] if first_call else [],
            }
        ],
        "global_findings": [],
        "needs_regeneration": first_call,
    }


def mock_regeneration_synth_requester(*, messages, prompt_config):
    return {
        "response": "Regenerated for this walkthrough.",
        "evidence_summary": "Regenerated evidence summary.",
        "bullets": (
            [
                {
                    "text": "Regenerated bullet citing the same allowed source.",
                    "citations": [
                        {"source_id": first_citation[0], "section": first_citation[1]}
                    ],
                }
            ]
            if first_citation
            else []
        ),
        "limitations": [],
    }


os.environ.setdefault("ANSWER_SYNTHESIS_OPENAI_API_KEY", "notebook-mock-key")
os.environ.setdefault("ANSWER_SYNTHESIS_OPENAI_MODEL", "notebook-mock-model")
mock_synthesizer = EvidenceAnswerSynthesizer(
    parameters=parameters,
    json_requester=mock_regeneration_synth_requester,
)

critic_enabled_parameters = parameters.__class__(
    **{**parameters.__dict__, "enable_answer_critic": True, "critic_max_regenerations": 1}
)

answer_b, critique_b, validation_b = finalize_answer_critique(
    query=QUERY,
    understanding=understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
    answer=draft_answer,
    contract=contract,
    validation=validation,
    synthesizer=mock_synthesizer,
    parameters=critic_enabled_parameters,
    critic_json_requester=forced_regeneration_critic,
)

print("(b) critic.enabled:", critique_b.enabled, "| source:", critique_b.source)
print("(b) critique.regenerated:", critique_b.regenerated)
print("(b) critic was called", regeneration_critic_calls["count"], "time(s) -- expect exactly 2 (initial + recheck)")
show_json(critique_b)
[b.text for b in answer_b.bullets]

(b) critic.enabled: True | source: llm
(b) critique.regenerated: True
(b) critic was called 2 time(s) -- expect exactly 2 (initial + recheck)


<IPython.core.display.JSON object>

['Regenerated bullet citing the same allowed source.']

## 13. Full service call — `QueryAnswerService.answer()`

Ties every step above together exactly as `POST /query-answer` does. Gated by `RUN_LIVE_ANSWER_SYNTHESIS` since it calls the real synthesizer (and the real critic, if `enable_answer_critic` is on).

In [17]:
if RUN_LIVE_ANSWER_SYNTHESIS:
    full_response = query_answer_service.answer(
        QUERY,
        openfda_limit=parameters.default_openfda_limit,
    )
    print("contract.coverage_level:", full_response.contract.coverage_level)
    print("validation.passed:", full_response.validation.passed)
    print("critique.enabled:", full_response.critique.enabled)
    show_json(full_response.critique)
else:
    print("Skipped: would call live answer synthesis. Set RUN_LIVE_ANSWER_SYNTHESIS = True in section 1.")

contract.coverage_level: direct
validation.passed: True
critique.enabled: False


<IPython.core.display.JSON object>

## 14. Gaps and things worth a second look

Observations from tracing this pipeline end to end while building this notebook — not blockers, but worth deciding on deliberately:

1. **[notebooks/03_backend_query_answer_walkthrough.ipynb](03_backend_query_answer_walkthrough.ipynb) is now stale.** It imports and calls `add_coverage_limitations` from `src/query_answer/coverage.py`, which D1 removed in favor of `contract.py` + `validation.py`. Running notebook 03 today raises an `ImportError` at the import cell. It needs an update (or should be retired) so it doesn't mislead anyone about the current pipeline shape.
2. **`critic.STRUCTURAL_CAVEAT_TOPICS` is a manually maintained allowlist.** It currently excludes only `interaction_terminology` from the "unresolved gap" check that downgrades `strong` → `partial`. If a future D1 caveat is added that's similarly always-emitted regardless of evidence quality (i.e. structural, not a real gap), it needs to be added here too or every claim sharing that intent will be capped at `partial` forever.
3. **`critic_max_regenerations` doesn't behave like a count.** `finalize_answer_critique` performs at most one regeneration via a single `if`, not a loop bounded by the parameter's value — so `critic_max_regenerations=2` behaves identically to `=1`; the parameter is really an on/off gate (`> 0`) today. Section 12(b) above demonstrates the actual behavior: the critic is called exactly twice (initial + one recheck) regardless of how high the parameter is set. Either the parameter should be renamed to reflect that it's a boolean-ish gate, or the orchestrator should actually loop up to that count.
4. **The critic's verdicts are trusted at face value.** `deterministic_support_status` only checks *structural* section membership (does a cited section belong to a contract-confirmed intent?); it never reads label text. The LLM critic is the only place that could catch a citation that's structurally valid but semantically unsupportive — and nothing downstream double-checks the critic's own `strong` verdicts against the cited text. A critic that hallucinates confidence has no backstop.
5. **`AnswerCritique.claims` can be a strict subset of `answer.bullets`.** If the critic skips a claim or returns an out-of-range/invalid index, `_parse_critique` silently drops it (see section 11's mock — it only returns one claim even though `floored_answer` may have more bullets). `answer.bullets[*].support_status` is always complete because the deterministic floor fills every bullet first and the critic only overlays the ones it actually addressed — but any consumer reading `critique.claims` directly (rather than the bullets) should not assume one entry per bullet.
6. **Only `bullets` get a per-claim audit.** The free-text `response` and `evidence_summary` fields are never individually checked for unsupported claims — `critique.global_findings` is the only backstop there, and it's advisory (nothing enforces it the way `validate_and_enforce` enforces `must_caveat`s). A model that smuggles an overreaching claim into `response` instead of a bullet would pass every guardrail here unflagged.

### Update — D2b resolved the cross-intent leakage this notebook surfaced

Running this notebook (and the web app) against *"Can I take ibuprofen for my migraine if I'm allergic to aspirin?"* surfaced a real bug: the section 10 output above used to show the interaction-gap bullet (the one stating no `drug_interactions` text was retrieved) flipping between `strong` and `limited` across runs, purely depending on which label section the LLM happened to cite for it. The cause was exactly what item 4 above touches on, but more severe: `deterministic_support_status` pooled `required_sections` across *every* addressed intent into one flat set (`critic._addressed_intent_sections`), so a citation that genuinely backed `allergy_context_check` (e.g. `warnings`) could spuriously "support" an unrelated `interaction_check` bullet just because both intents happened to share a section name.

**Fix (D2b):** each bullet now carries a `topic` (`EvidenceBullet.topic`), parsed from synthesis output and whitelisted against the contract's own topics — the model can select an existing topic but never invent one, same pattern as citation whitelisting. `deterministic_support_status` scopes its section check to that topic's own `required_sections` when a match exists (`critic._topic_required_sections`, exercised in section 10's inspection cell above), falling back to the legacy pooled check only for untagged bullets. `build_answer_contract` also now uses each intent's actually-matched section(s) instead of the static `INTENT_REQUIRED_SECTIONS` allow-list. Re-running the live cells above against the same query now consistently scores that bullet `none`/`limited` regardless of which section gets cited, while the allergy bullets citing `warnings`/`contraindications` correctly remain `strong`.

This doesn't change item 4's underlying point — the floor is still purely structural (it never reads label text); the LLM critic remains the only place that could catch a citation that's structurally valid but semantically unsupportive. D2b closes the specific cross-intent false-positive, not the broader "no semantic re-check" gap.

### Update — D2c fixed an evidence-packet truncation bug this notebook also surfaced

Even after D2b, the same query kept producing a bullet saying the packet didn't include any `drug_interactions` text for ibuprofen — despite the contract (section 6 above) marking `interaction_check` "addressed." Digging into section 7's evidence packet showed why: `label_section_payloads()` walked sections in a fixed priority order (`boxed_warning`, `contraindications`, `warnings`, `drug_interactions`, ...) with a flat 16-entry cap per drug, no per-section sub-limit, and no deduplication. Ibuprofen alone had 2 `boxed_warning` + 2 `contraindications` + 13 `warnings` entries (10 of which were near-identical "allergy alert" boilerplate repeated across different manufacturer label records) = 17 non-empty entries — already past the cap before `drug_interactions` (2 real entries) was ever reached. The deterministic coverage layer (`intent_evidence_status`, section 5) correctly found that evidence on the *untruncated* dossier data, so the contract promised it — but the LLM never actually saw it, because evidence-packet construction truncated independently with no awareness of what the contract relied on.

**Fix (D2c):** `label_section_payloads()` now deduplicates near-identical boilerplate per section before counting it against any cap, caps each section to `MAX_SECTION_ENTRIES` so one bloated section can't crowd out the rest, and guarantees any section the contract relies on (`required_label_sections(contract)`, now shared between `contract.py` and `critic.py` instead of duplicated) survives the overall cap — evicting from the lowest-priority tail if it was already full. Section 7 above now shows `drug_interactions` present for both ibuprofen and aspirin, and a live re-run of this query no longer produces the false "no drug_interactions text" bullet; it now cites the real retrieved aspirin/ibuprofen interaction text instead.

Like D2b, this doesn't replace the still-open structural-only nature of the deterministic floor (item 4) — it fixes a retrieval/packet-construction bug that was starving the floor and the LLM of evidence that was always there.